In [0]:
%python
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%python
spark.sql("""
          CREATE TABLE IF NOT EXISTS movie_gold.results_movie
          (
              year_release_date INT,
              country_name STRING,
              company_name STRING,
              budget DOUBLE,
              revenue DOUBLE,
              movie_id INT,
              country_id INT,
              company_id INT,
              created_date DATE,
              updated_date DATE
          )
          USING DELTA
          """)

DataFrame[]

In [0]:
%python
spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW v_results_movie
            AS
            SELECT m.year_release_date, c.country_name, pco.company_name, m.budget, m.revenue, m.movie_id, c.country_id, pco.company_id
            FROM movie_silver.movies m
            INNER JOIN movie_silver.productions_countries pc ON m.movie_id = pc.movie_id
            INNER JOIN movie_silver.countries c ON pc.country_id = c.country_id
            INNER JOIN movie_silver.movies_companies mc on m.movie_id = mc.movie_id
            INNER JOIN movie_silver.productions_companies pco ON mc.company_id = pco.company_id
            WHERE m.file_date = '{v_file_date}'
        """)

DataFrame[]

In [0]:
MERGE INTO movie_gold.results_movie tgt
USING v_results_movie src
ON (tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id)
WHEN MATCHED THEN
    UPDATE SET
        tgt.year_release_date = src.year_release_date,
        tgt.country_name = src.country_name,
        tgt.company_name = src.company_name,
        tgt.budget = src.budget,
        tgt.revenue = src.revenue,
        tgt.updated_date = current_timestamp
WHEN NOT MATCHED THEN
    INSERT (year_release_date, country_name, company_name, budget, revenue, movie_id, country_id, company_id, created_date) VALUES (src.year_release_date, src.country_name, src.company_name, src.budget, src.revenue, src.movie_id, src.country_id, src.company_id, current_timestamp)
       

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
14386,0,0,14386


In [0]:
SELECT * FROM movie_gold.results_movie

year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id,created_date,updated_date
1995,United States of America,A Band Apart,4000000.0,4300000.0,5,214,59,2026-09-15,null
1995,United States of America,Miramax Films,4000000.0,4300000.0,5,214,14,2026-09-15,null
1977,United States of America,Twentieth Century Fox Film Corporation,1.1E7,7.75398007E8,11,214,306,2026-09-15,null
1977,United States of America,Lucasfilm,1.1E7,7.75398007E8,11,214,1,2026-09-15,null
2003,United States of America,Pixar Animation Studios,9.4E7,9.40335536E8,12,214,3,2026-09-15,null
1994,United States of America,Paramount Pictures,5.5E7,6.77945399E8,13,214,4,2026-09-15,null
1999,United States of America,Jinks/Cohen Company,1.5E7,3.56296601E8,14,214,2721,2026-09-15,null
1999,United States of America,DreamWorks SKG,1.5E7,3.56296601E8,14,214,27,2026-09-15,null
2000,Argentina,Film4,1.28E7,4.0031879E7,16,131,9349,2026-09-15,null
2000,Argentina,Vrijzinnig Protestantse Radio Omroep (VPRO),1.28E7,4.0031879E7,16,131,8659,2026-09-15,null
